# Notebook 2 – Adult Diabetes Data

## Objective
Download, clean, quality check, and export age-standardized adult diabetes prevalence data for Pacific Island jurisdictions.

### Source
NCD Risk Factor Collaboration (NCD-RisC), country-specific age-standardized diabetes estimates.

### Output
processed_data/adult_diabetes.csv

## 1. Initialize project and download source data

In [5]:
import pandas as pd
import requests
from io import StringIO
import os
import time

os.makedirs("raw_data/NCD_RisC", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

url = "https://www.ncdrisc.org/downloads/dm-2024/NCD_RisC_Lancet_2024_Diabetes_age_standardised_countries.csv"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Encoding": "identity"
}

raw_file = "raw_data/NCD_RisC/ncdrisc_diabetes_age_standardised_countries_raw.csv"

for attempt in range(1, 8):
    try:
        print(f"Attempt {attempt}...")
        response = requests.get(url, headers=headers, timeout=180, stream=True)
        print("Status code:", response.status_code)
        response.raise_for_status()

        with open(raw_file, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print("Downloaded MB:", round(os.path.getsize(raw_file) / 1_000_000, 2))

        diabetes_raw = pd.read_csv(raw_file)

        print("Rows downloaded:", len(diabetes_raw))
        print("Columns:")
        print(diabetes_raw.columns.tolist())

        display(diabetes_raw.head())
        break

    except Exception as e:
        print("Failed:", e)
        if attempt == 7:
            raise
        time.sleep(5)

Attempt 1...
Status code: 200
Downloaded MB: 2.1
Rows downloaded: 13200
Columns:
['Country/Region/World', 'ISO', 'Sex', 'Year', 'Age', 'Prevalence of diabetes (18+ years)', 'Prevalence of diabetes (18+ years) lower 95% uncertainty interval', 'Prevalence of diabetes (18+ years) upper 95% uncertainty interval', 'Proportion of people with diabetes who were treated (30+ years)', 'Proportion of people with diabetes who were treated (30+ years) lower 95% uncertainty interval', 'Proportion of people with diabetes who were treated (30+ years) upper 95% uncertainty interval']


,Country/Region/World,ISO,Sex,Year,Age,Prevalence of diabetes (18+ years),Prevalence of diabetes (18+ years) lower 95% uncertainty interval,Prevalence of diabetes (18+ years) upper 95% uncertainty interval,Proportion of people with diabetes who were treated (30+ years),Proportion of people with diabetes who were treated (30+ years) lower 95% uncertainty interval,Proportion of people with diabetes who were treated (30+ years) upper 95% uncertainty interval
0,Afghanistan,AFG,Men,1990,Age-standardised,0.102185,0.043214,0.185308,0.234837,0.084755,0.464630
1,Afghanistan,AFG,Men,1991,Age-standardised,0.104102,0.044941,0.189061,0.234642,0.087199,0.457768
2,Afghanistan,AFG,Men,1992,Age-standardised,0.106136,0.046302,0.192939,0.234537,0.088188,0.451526
3,Afghanistan,AFG,Men,1993,Age-standardised,0.108260,0.047838,0.195853,0.234461,0.090243,0.448084
4,Afghanistan,AFG,Men,1994,Age-standardised,0.110458,0.048876,0.198207,0.234399,0.092058,0.440310


## 2. Filter to study population

In [6]:
pacific_iso3 = [
    "ASM", "COK", "FJI", "FSM", "PYF",
    "KIR", "MHL", "NRU", "NIU",
    "PLW", "WSM", "SLB", "TKL",
    "TON", "TUV", "VUT"
]

diabetes_pacific = diabetes_raw[
    diabetes_raw["ISO"].isin(pacific_iso3)
].copy()

diabetes_pacific = diabetes_pacific.rename(columns={
    "Country/Region/World": "Country",
    "ISO": "ISO3",
    "Prevalence of diabetes (18+ years)": "Diabetes_Pct",
    "Proportion of people with diabetes who were treated (30+ years)": "Diabetes_Treated_Pct"
})

diabetes_pacific = (
    diabetes_pacific
    .groupby(["ISO3", "Country", "Year"], as_index=False)
    .agg({
        "Diabetes_Pct": "mean",
        "Diabetes_Treated_Pct": "mean"
    })
)

print("Rows:", len(diabetes_pacific))
diabetes_pacific.head(20)

Rows: 528


,ISO3,Country,Year,Diabetes_Pct,Diabetes_Treated_Pct
0,ASM,American Samoa,1990,0.211932,0.274700
1,ASM,American Samoa,1991,0.220034,0.271805
2,ASM,American Samoa,1992,0.228339,0.268938
3,ASM,American Samoa,1993,0.236888,0.266083
4,ASM,American Samoa,1994,0.245686,0.263203
5,ASM,American Samoa,1995,0.254746,0.260308
6,ASM,American Samoa,1996,0.264017,0.257523
7,ASM,American Samoa,1997,0.273364,0.254844
8,ASM,American Samoa,1998,0.282665,0.252340
9,ASM,American Samoa,1999,0.291793,0.250142


## 3. Quality assurance

In [7]:
print(diabetes_pacific.info())

print()

print("Countries:")
print(sorted(diabetes_pacific["Country"].unique()))

print()

print("Years:")
print(diabetes_pacific["Year"].min(), "to", diabetes_pacific["Year"].max())

print()

print("Missing diabetes values:")
print(diabetes_pacific["Diabetes_Pct"].isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 528 entries, 0 to 527
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ISO3                  528 non-null    object 
 1   Country               528 non-null    object 
 2   Year                  528 non-null    int64  
 3   Diabetes_Pct          528 non-null    float64
 4   Diabetes_Treated_Pct  528 non-null    float64
dtypes: float64(2), int64(1), object(2)
memory usage: 20.8+ KB
None

Countries:
['American Samoa', 'Cook Islands', 'Federated States of Micronesia', 'Fiji', 'French Polynesia', 'Kiribati', 'Marshall Islands', 'Nauru', 'Niue', 'Palau', 'Samoa', 'Solomon Islands', 'Tokelau', 'Tonga', 'Tuvalu', 'Vanuatu']

Years:
1990 to 2022

Missing diabetes values:
0


## 4. Coverage assessment

In [8]:
coverage = (
    diabetes_pacific
    .groupby(["ISO3", "Country"])
    .agg(
        First_Year=("Year", "min"),
        Last_Year=("Year", "max"),
        Years=("Year", "count")
    )
    .reset_index()
)

coverage

,ISO3,Country,First_Year,Last_Year,Years
0,ASM,American Samoa,1990,2022,33
1,COK,Cook Islands,1990,2022,33
2,FJI,Fiji,1990,2022,33
3,FSM,Federated States of Micronesia,1990,2022,33
4,KIR,Kiribati,1990,2022,33
5,MHL,Marshall Islands,1990,2022,33
6,NIU,Niue,1990,2022,33
7,NRU,Nauru,1990,2022,33
8,PLW,Palau,1990,2022,33
9,PYF,French Polynesia,1990,2022,33


## 5. Export cleaned dataset

In [9]:
diabetes_pacific.to_csv(
    "processed_data/adult_diabetes.csv",
    index=False
)

coverage.to_csv(
    "processed_data/adult_diabetes_coverage.csv",
    index=False
)

print("Notebook 2 complete.")
print("Files saved:")
print("- processed_data/adult_diabetes.csv")
print("- processed_data/adult_diabetes_coverage.csv")

Notebook 2 complete.
Files saved:
- processed_data/adult_diabetes.csv
- processed_data/adult_diabetes_coverage.csv


In [10]:
print(diabetes_pacific.info())

print()

print("Countries:")
print(sorted(diabetes_pacific["Country"].unique()))

print()

print("Years:")
print(diabetes_pacific["Year"].min(), "to", diabetes_pacific["Year"].max())

print()

print("Missing values by column:")
print(diabetes_pacific.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 528 entries, 0 to 527
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ISO3                  528 non-null    object 
 1   Country               528 non-null    object 
 2   Year                  528 non-null    int64  
 3   Diabetes_Pct          528 non-null    float64
 4   Diabetes_Treated_Pct  528 non-null    float64
dtypes: float64(2), int64(1), object(2)
memory usage: 20.8+ KB
None

Countries:
['American Samoa', 'Cook Islands', 'Federated States of Micronesia', 'Fiji', 'French Polynesia', 'Kiribati', 'Marshall Islands', 'Nauru', 'Niue', 'Palau', 'Samoa', 'Solomon Islands', 'Tokelau', 'Tonga', 'Tuvalu', 'Vanuatu']

Years:
1990 to 2022

Missing values by column:
ISO3                    0
Country                 0
Year                    0
Diabetes_Pct            0
Diabetes_Treated_Pct    0
dtype: int64


# Notebook Summary

This notebook successfully:

- Downloaded NCD-RisC age-standardized adult diabetes data.
- Filtered to the 16 Pacific Island jurisdictions.
- Averaged male and female estimates to create a single country-year dataset.
- Produced two diabetes indicators:
  - Adult diabetes prevalence
  - Diabetes treatment coverage
- Verified complete coverage from 1990–2022 with no missing values.
- Exported the cleaned dataset for downstream analysis.